<a href="https://colab.research.google.com/github/Amazeble/tts-dataset-generator/blob/main/colab/tts-dataset-generator-colab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
#@title ⚙️ STEP 0: Configuration

#@markdown ### 📂 Project Settings
project_name = "Adriene" #@param {type:"string"}
base_directory = "MyTTSDataset" #@param {type:"string"}
input_file_path = "/content/drive/MyDrive/Chatterbox/Adriene.wav" #@param {type:"string"}

#@markdown ### 🔊 Audio Segmentation Settings
min_duration = 3.0 #@param {type:"number"}
max_duration = 11 #@param {type:"number"}
silence_threshold = -40 #@param {type:"integer"}
min_silence_len = 250 #@param {type:"integer"}
keep_silence = 150 #@param {type:"integer"}
sample_rate = 22050 #@param {type:"integer"}

#@markdown ### 🎙️ Transcription Settings
whisper_model = "large" #@param ["tiny", "base", "small", "medium", "large"]
language = "en" #@param {type:"string"}
ljspeech = True #@param {type:"boolean"}

#@markdown ### 📝 Logging Settings
log_level = "INFO" #@param ["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"]

# Create config dictionary
config = {
    "project_name": project_name,
    "base_directory": base_directory,
    "input_file_path": input_file_path,
    "min_duration": min_duration,
    "max_duration": max_duration,
    "silence_threshold": silence_threshold,
    "min_silence_len": min_silence_len,
    "keep_silence": keep_silence,
    "sample_rate": sample_rate,
    "whisper_model": whisper_model,
    "language": language,
    "ljspeech": ljspeech,
    "log_level": log_level
}

# Save config to JSON file
import json
config_file_path = "config.json"
with open(config_file_path, "w") as f:
    json.dump(config, f, indent=2)

print("="*50)
print("✅ CONFIGURATION SUMMARY")
print("="*50)
print(f"Project Name: {project_name}")
print(f"Base Directory: {base_directory}")
print(f"Input File: {input_file_path}")
print(f"Whisper Model: {whisper_model}")
print(f"Language: {language}")
print(f"Sample Rate: {sample_rate}")
print(f"LJSpeech Format: {ljspeech}")
print("="*50)
print(f"\n💾 Config saved to: {config_file_path}")
print("\n📋 Note: Run STEP 2 after cloning to sync config to config.py")

✅ CONFIGURATION SUMMARY
Project Name: Adriene
Input File: /content/drive/MyDrive/Chatterbox/Adriene.wav
Whisper Model: large
Language: en
Sample Rate: 22050
LJSpeech Format: True

💾 Config saved to: config.json


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Clone this repository
!git clone https://github.com/amazeble/tts-dataset-generator.git

Cloning into 'tts-dataset-generator'...
remote: Enumerating objects: 726, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 726 (delta 75), reused 69 (delta 43), pack-reused 592 (from 3)
Receiving objects: 100% (726/726), 84.85 MiB | 25.04 MiB/s, done.
Resolving deltas: 100% (84/84), done.


In [4]:
%cd tts-dataset-generator

# Sync config.json to config.py by updating the Config class
import json
import os

# Check if config.json exists from STEP 0
if os.path.exists('../config.json'):
    # Load config from JSON
    with open('../config.json', 'r') as f:
        config_data = json.load(f)
    
    # Read existing config.py
    with open('config.py', 'r') as f:
        config_py_content = f.read()
    
    # Update the Config class __init__ method to load from JSON
    import re
    
    new_init_method = '''    def __init__(self):
        # Load from JSON if available (from Colab STEP 0)
        json_path = os.path.join(os.path.dirname(__file__), '..', 'config.json')
        if os.path.exists(json_path):
            with open(json_path, 'r') as f:
                data = json.load(f)
            self.project_name = data.get('project_name', 'MyProject')
            self.base_directory = data.get('base_directory', 'MyTTSDataset')
            self.input_file_path = data.get('input_file_path', '')
            self.min_duration = data.get('min_duration', 3.0)
            self.max_duration = data.get('max_duration', 10.0)
            self.silence_threshold = data.get('silence_threshold', -40)
            self.min_silence_len = data.get('min_silence_len', 250)
            self.keep_silence = data.get('keep_silence', 150)
            self.sample_rate = data.get('sample_rate', 22050)
            self.whisper_model = data.get('whisper_model', 'large')
            self.language = data.get('language', 'en')
            self.ljspeech = data.get('ljspeech', True)
            self.log_level = data.get('log_level', 'INFO')
        else:
            # Default values
            self.project_name = 'MyProject'
            self.base_directory = 'MyTTSDataset'
            self.input_file_path = ''
            self.min_duration = 3.0
            self.max_duration = 10.0
            self.silence_threshold = -40
            self.min_silence_len = 250
            self.keep_silence = 150
            self.sample_rate = 22050
            self.whisper_model = 'large'
            self.language = 'en'
            self.ljspeech = True
            self.log_level = 'INFO'''
    
    # Replace the __init__ method in the Config class
    pattern = r'(\s+def __init__\(self\):.*?)(?=\n\s+def |\nclass |$)'
    updated_content = re.sub(pattern, new_init_method, config_py_content, flags=re.DOTALL)
    
    with open('config.py', 'w') as f:
        f.write(updated_content)
    
    print("✅ Config synced from config.json to existing config.py")
else:
    print("⚠️ No config.json found. Using existing config.py")

/content/tts-dataset-generator


In [3]:
# Install dependencies
!apt update
!apt install ffmpeg
!pip install torch==2.6.0 torchaudio==2.6.0 torchvision==0.21.0  pydub natsort --index-url https://download.pytorch.org/whl/cu124

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://cli.github.com/packages stable InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
145 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as r

In [2]:
!pip install faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 99.0 MB/s eta 0:00:00


In [6]:
!pip install moviepy==2.1.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.7/126.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 89.6 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: moviepy
    Found existing installation: moviepy 1.0.3
    Uninstalling moviepy-1.0.3:
      Successfully uninstalled moviepy-1.0.3


In [9]:
!pip install ffmpeg-python

In [6]:
%cd tts-dataset-generator/


/content/tts-dataset-generator


In [ ]:
# Run main.py using the config.json file created in STEP 0
# This cell reads the configuration from the JSON file

import json

# Read config from JSON file
with open("config.json", "r") as f:
    config = json.load(f)

print("Loading configuration from config.json...")
print("="*50)
print("✅ LOADED CONFIGURATION")
print("="*50)
for key, value in config.items():
    print(f"{key}: {value}")
print("="*50)

# Build the command string from loaded config
cmd_parts = [
    "python main.py",
    f"--file \"{config['input_file_path']}\"",
    f"--project \"{config['project_name']}\"",
    f"--min-duration {config['min_duration']}",
    f"--max-duration {config['max_duration']}",
    f"--silence-threshold {config['silence_threshold']}",
    f"--min-silence-len {config['min_silence_len']}",
    f"--keep-silence {config['keep_silence']}",
    f"--model {config['whisper_model']}",
    f"--language {config['language']}",
    f"--ljspeech {config['ljspeech']}",
    f"--sample_rate {config['sample_rate']}",
    f"--log-level {config['log_level']}"
]

command = " ".join(cmd_parts)
print(f"\n🚀 Executing: {command}\n")
!{command}

Loading configuration from config.json...
✅ LOADED CONFIGURATION
project_name: Adriene
input_file_path: /content/drive/MyDrive/Chatterbox/Adriene.wav
min_duration: 3.0
max_duration: 11
silence_threshold: -40
min_silence_len: 250
keep_silence: 150
sample_rate: 22050
whisper_model: large
language: en
ljspeech: True
log_level: INFO

🚀 Executing: python main.py --file "/content/drive/MyDrive/Chatterbox/Adriene.wav" --project "Adriene" --min-duration 3.0 --max-duration 11 --silence-threshold -40 --min-silence-len 250 --keep-silence 150 --model large --language en --ljspeech True --sample_rate 22050 --log-level INFO


    ╔════════════════════════════════════════════════════════════╗
    ║                                                            ║
    ║        Audio/Video Segmentation & Transcription Tool       ║
    ║                                                            ║
    ╚════════════════════════════════════════════════════════════╝
    
2026-07-18 21:00:08,357 - INFO - Running

In [15]:
import os

# List files in the Chatterbox directory to find the correct filename
directory_path = "/content/drive/MyDrive/Chatterbox"
if os.path.exists(directory_path):
    print(f"Files in {directory_path}:")
    for f in os.listdir(directory_path):
        print(f"- {f}")
else:
    print(f"Directory not found: {directory_path}")

Files in /content/drive/MyDrive/Chatterbox:
- Adriene.wav
- chatterbox_output
- MyTTSDataset
- Download (16).wav
